# 第 1 課：訓練一個會認手寫數字的 CNN

這份 notebook 的目的**不是**拿到高準確率 —— MNIST 隨便練都有 99%。
目的是讓你看見「訓練」這件事到底在做什麼，以及**哪些設計是被 AMB82 硬體逼出來的**。

用法：`Runtime -> Run all` 可以一路跑完，但**建議一格一格跑**（Shift+Enter），
每一格跑完先看輸出、讀下面的說明，再往下。

跑完會產出兩個東西，是第 2 課（轉檔）的輸入：
- `mnist_cnn.h5` —— 訓練好的浮點模型
- `mnist_calib.zip` —— 200 張校正圖 + `dataset.txt`


## 0. 環境確認

先確認 TF 版本。Colab 現在是 TF 2.19 / **Keras 3**。

Keras 3 存出來的 `.h5` 內部 schema 跟 Keras 2 不一樣，而 Realtek 的 acuity 轉檔工具
是 Keras 2 時代的東西。**訓練本身完全不受影響**，只有最後存檔那一步要注意 ——
我們到第 9 格再處理，先不要為它分心。


In [ ]:
import os, sys, math, zipfile
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print("python     :", sys.version.split()[0])
print("tensorflow :", tf.__version__)
print("keras      :", tf.keras.__version__)
print("GPU        :", tf.config.list_physical_devices('GPU') or "沒有（沒關係，這個模型用 CPU 也是幾十秒）")

# 固定亂數種子。訓練有隨機性（初始權重、資料順序），固定種子你每次跑才會
# 得到一樣的數字，才有辦法說「我改了 X，結果變好了」。不固定就分不出
# 是你的改動有效還是運氣好。
SEED = 1234
tf.keras.utils.set_random_seed(SEED)

## 1. 先看資料

機器學習第一件事永遠是**把資料印出來看**，不是寫模型。
你要知道四件事：有幾筆、什麼形狀、數值範圍、每個類別數量平不平均。

這四件任何一件搞錯，後面整條鏈都是錯的，**而且不會報錯**。


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print("訓練集 :", x_train.shape, x_train.dtype, " 標籤:", y_train.shape)
print("測試集 :", x_test.shape,  x_test.dtype,  " 標籤:", y_test.shape)
print("像素值範圍 :", x_train.min(), "~", x_train.max())
print()
print("每個類別幾張（要大致平均，不然模型會偏心）:")
for d in range(10):
    n = int((y_train == d).sum())
    print("  數字 %d : %5d 張  %s" % (d, n, "#" * (n // 150)))

### 用文字把一張圖印出來

`28x28` 到底是什麼？直接看。下面把像素值畫成文字 ——
手寫數字在電腦眼裡就是一個 28x28 的亮度矩陣，**背景是 0，筆畫是 255**。

記住這個「背景黑、筆畫白」的極性。第 4 課在板子上餵真實影像時，
如果你拿的是白紙黑字，就得自己反過來，不然模型看到的是它沒學過的東西。


In [ ]:
def show_ascii(img, label=None):
    chars = " .:-=+*#%@"
    print("label =", label)
    for row in img:
        print("".join(chars[min(int(v) * len(chars) // 256, len(chars)-1)] for v in row))

show_ascii(x_train[0], y_train[0])

In [ ]:
# 再用圖看 10 張，順便確認標籤真的對得上圖
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title("label = %d" % y_train[i])
    ax.axis("off")
plt.tight_layout(); plt.show()

## 2. 輸入契約 —— 這一格是整條鏈最重要的地方

「輸入契約」= 模型**期待**收到什麼樣的數字。訓練時餵什麼，裝置上就必須餵一模一樣的東西。

這裡有兩個決定，**都不是機器學習的考量，是被 AMB82 韌體逼出來的**：

### ① 三通道，不是一通道

MNIST 是灰階，理論上 `28x28x1` 就夠。但 AMB82 韌體裡的 `img_resize_planar()`
永遠寫 `W*H*3` bytes —— 它的 `img_t` 結構根本沒有 channel 欄位，通道數寫死是 3。

你如果訓練單通道模型，裝置端會往一個只有 784 bytes 的 buffer 裡塞 2352 bytes，
直接踩壞旁邊的記憶體。**所以我們把灰階複製成三份 R=G=B。**

> 教訓：模型的輸入形狀要遷就韌體 API 的形狀，不是反過來。

### ② 訓練時除以 255，裝置端不除

訓練時餵 `0.0 ~ 1.0` 的 float（梯度下降在小數值域比較穩）。
但相機給韌體的是 `0 ~ 255` 的 uint8。

這個落差**不在訓練裡解決**，而是留到第 2 課的 `inputmeta.yml` 用
`scale = 1/255 = 0.00392157` 去橋接。

> 這就是為什麼 inputmeta 寫錯會全盤皆錯：它是訓練世界和硬體世界之間**唯一**的翻譯層，
> 而且寫錯的時候每一步都會「成功」，只是答案全是垃圾。


In [ ]:
CH = 3

def to3ch(x):
    """(N,28,28) 灰階 -> (N,28,28,3)，三個通道放一樣的值。"""
    return np.repeat(x[..., None], CH, axis=-1)

x_train_f = to3ch(x_train).astype("float32") / 255.0
x_test_f  = to3ch(x_test ).astype("float32") / 255.0

print("訓練張量 :", x_train_f.shape, x_train_f.dtype)
print("數值範圍 :", x_train_f.min(), "~", x_train_f.max())
print("三個通道一樣嗎 :", np.array_equal(x_train_f[..., 0], x_train_f[..., 2]))

# 這個 scale 常數等一下要抄進 inputmeta.yml，先印出來記著
print()
print("inputmeta 要用的 scale = 1/255 =", 1.0 / 255.0)

## 3. 模型架構

逐層看它在做什麼。輸入 `28x28x3`：

| 層 | 輸出形狀 | 在做什麼 |
|---|---|---|
| `Conv2D(16, 3x3)` | 28x28x16 | 16 個 3x3 小濾鏡掃過整張圖，各自找一種局部花樣（邊緣、轉角…） |
| `MaxPool(2)` | 14x14x16 | 每 2x2 取最大值。解析度減半 → 之後每個像素「看得更廣」，計算量也少 4 倍 |
| `Conv2D(32, 3x3)` | 14x14x32 | 把前一層找到的花樣**組合**成更複雜的（弧線、閉環…） |
| `MaxPool(2)` | 7x7x32 | 再減半 |
| `Reshape(1568)` | 1568 | 攤平成一條向量 |
| `Dense(64)` | 64 | 全連接：1568 個特徵壓成 64 個 |
| `Dense(10)` | 10 | 每個數字一個分數（logit） |
| `Softmax` | 10 | 分數轉成加起來 = 1 的機率 |

### 為什麼是 `Reshape` 不是 `Flatten`？

`Flatten` 在計算圖上會產生一個**動態 shape** 的節點 —— 輸出大小要等執行時才知道。
NPU 的編譯器必須在**編譯期**就把每一塊記憶體配死，遇到動態 shape 直接轉不出來。

所以我們自己把 `7*7*32 = 1568` 算好寫成常數。同理 **不能用** `GlobalAveragePooling`。

> 教訓：NPU 的世界裡沒有「執行時才決定」這種事。

### 為什麼不加 Data Augmentation 層？

`RandomRotation` / `RandomTranslation` 這種層會被一起存進 `.h5`，轉檔工具不認得。
要做資料增強，就在 `tf.data` 那一層做，不要放進模型裡。


In [ ]:
SIZE, NUM_CLASSES = 28, 10
FLAT = (SIZE // 4) * (SIZE // 4) * 32     # 7*7*32 -- 自己算好的常數，不用 Flatten
print("FLAT =", FLAT)

def build_model():
    return tf.keras.Sequential([
        tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu",
                               input_shape=(SIZE, SIZE, CH), name="conv1"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu", name="conv2"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Reshape((FLAT,)),
        tf.keras.layers.Dense(64, activation="relu", name="dense1"),
        tf.keras.layers.Dense(NUM_CLASSES, name="dense2"),
        tf.keras.layers.Softmax(),
    ])

model = build_model()
model.summary()

### 自己驗算參數量

不要看到 `summary()` 就算了，自己算一次才知道參數是怎麼來的。
一層 Conv 的權重數 = `kernel_h * kernel_w * 輸入通道 * 輸出通道`，再加 `輸出通道` 個 bias。


In [ ]:
checks = [
    ("conv1",  3*3*3*16 + 16),
    ("conv2",  3*3*16*32 + 32),
    ("dense1", FLAT*64 + 64),
    ("dense2", 64*10 + 10),
]
total = 0
for name, mine in checks:
    keras_n = int(sum(np.prod(w.shape) for w in model.get_layer(name).get_weights()))
    total += keras_n
    print("%-7s 我算的 %8d   keras %8d   %s" % (name, mine, keras_n,
                                               "OK" if mine == keras_n else "對不上!"))
print("-" * 48)
print("總計 %d 個參數 (float32 = %.1f KB，量化成 uint8 後約 %.1f KB)"
      % (total, total*4/1024, total/1024))
print()
print("注意 dense1 一層就佔了 %.0f%% 的參數 -- 參數幾乎都在全連接層。"
      % (100.0*(FLAT*64+64)/total))
print("但等一下你會發現計算時間幾乎都在 conv 層。參數多 != 算得久，這兩件事是分開的。")

## 4. 起跑線：訓練前它有多笨？

**這一格是整份 notebook 我最希望你養成的習慣。**

訓練之前先量一次。權重現在是亂數，所以模型應該是在瞎猜：

- 準確率應該 ≈ `1/10 = 0.10`
- loss 應該 ≈ `ln(10) = 2.3026`

為什麼是 `ln(10)`？cross-entropy loss = `-ln(給正解的機率)`。
完全瞎猜時每個類別給 `1/10`，所以 `-ln(0.1) = 2.3026`。

**有了這條起跑線，你才有辦法判斷訓練到底有沒有在學。**
如果練半天 loss 還卡在 2.30，那不是「學得慢」，是**根本沒在學**
（學習率爆掉、標籤對錯、輸入全 0…），該去除錯而不是多練幾輪。


In [ ]:
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

CHANCE = math.log(NUM_CLASSES)
loss0, acc0 = model.evaluate(x_test_f, y_test, verbose=0)
print("訓練前 test loss = %.4f   （瞎猜基準 ln(10) = %.4f）" % (loss0, CHANCE))
print("訓練前 test acc  = %.4f   （瞎猜基準 1/10   = 0.1000）" % acc0)

# 把訓練前的 conv1 權重存起來，第 6 格要拿來跟訓練後對照
W0_conv1 = model.get_layer("conv1").get_weights()[0].copy()
print()
print("conv1 權重形狀 =", W0_conv1.shape, "（3x3 kernel、3 個輸入通道、16 個輸出 filter）")

## 5. 訓練

三個東西先講清楚：

- **optimizer = `adam`** —— 決定「知道該往哪改之後，要改多大步」。Adam 會自動幫每個
  參數調步伐，是現在的預設選擇。
- **loss = `sparse_categorical_crossentropy`** —— 衡量「錯多少」的尺。`sparse` 是說標籤
  直接給整數 `3`，不用轉成 one-hot `[0,0,0,1,0,...]`。
- **`validation_split=0.1`** —— 從訓練資料切 10% 出來**不拿去訓練**，只用來檢查。
  這是你唯一能看出**過擬合**的方法：訓練準確率一直漲但驗證準確率不漲 = 模型在背答案。

`verbose=2` 讓它每個 epoch 印一行。**一行一行看**：

- 第 1 個 epoch 結束時 loss 就會從 2.30 掉到 0.2 附近 —— 大部分的學習發生在最開始
- 後面幾個 epoch 是在磨細節，進步越來越小


In [ ]:
EPOCHS = 6
BATCH_SIZE = 128

history = model.fit(x_train_f, y_train,
                    epochs=EPOCHS, batch_size=BATCH_SIZE,
                    validation_split=0.1, verbose=2)

loss, acc = model.evaluate(x_test_f, y_test, verbose=0)
print()
print("=" * 54)
print("float test accuracy = %.4f   (訓練前是 %.4f)" % (acc, acc0))
print("float test loss     = %.4f   (訓練前是 %.4f)" % (loss, loss0))
print("=" * 54)

### 把學習曲線畫出來

看兩條線的**差距**，不是只看高度：

- 兩條貼在一起 → 模型還有餘力，可以練更久或做更大
- 訓練線一直漲、驗證線持平甚至下滑 → **過擬合**，模型在背訓練集，該停了


In [ ]:
h = history.history
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))

a1.plot(h["loss"], "o-", label="train")
a1.plot(h["val_loss"], "s-", label="validation")
a1.axhline(CHANCE, ls="--", c="r", label="chance ln(10)")
a1.set_xlabel("epoch"); a1.set_ylabel("loss"); a1.set_title("loss")
a1.legend(); a1.grid(alpha=.3)

a2.plot(h["accuracy"], "o-", label="train")
a2.plot(h["val_accuracy"], "s-", label="validation")
a2.axhline(0.1, ls="--", c="r", label="chance 1/10")
a2.set_xlabel("epoch"); a2.set_ylabel("accuracy"); a2.set_title("accuracy")
a2.legend(); a2.grid(alpha=.3)

plt.tight_layout(); plt.show()

# 每個 epoch 的完整數字表
print("%-6s %10s %10s %10s %10s %10s" %
      ("epoch", "train loss", "val loss", "train acc", "val acc", "val 進步"))
print("-" * 62)
print("%-6s %10.4f %10s %10.4f %10s %10s" % ("(訓練前)", loss0, "-", acc0, "-", "-"))
for i in range(len(h["loss"])):
    d = h["val_accuracy"][i] - (h["val_accuracy"][i-1] if i else acc0)
    print("%-6d %10.4f %10.4f %10.4f %10.4f %+10.4f" %
          (i+1, h["loss"][i], h["val_loss"][i],
           h["accuracy"][i], h["val_accuracy"][i], d))
print("-" * 62)
print("最終 test acc = %.4f" % acc)

## 6. 它到底學到了什麼？

`conv1` 有 16 個 `3x3x3` 的 filter。訓練前是亂數，訓練後應該變成**有結構的花樣** ——
你會看到一邊亮一邊暗的（偵測邊緣）、中間亮周圍暗的（偵測點）等等。

這是「CNN 在學什麼」最直接的證據。不要跳過這格。


In [ ]:
W1 = model.get_layer("conv1").get_weights()[0]   # (3,3,3,16)

def draw_filters(W, title):
    fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
    fig.suptitle(title)
    for i, ax in enumerate(axes.ravel()):
        f = W[..., i]
        f = (f - f.min()) / (f.max() - f.min() + 1e-9)   # 正規化到 0~1 才看得見
        ax.imshow(f, interpolation="nearest")
        ax.set_title("#%d" % i, fontsize=8); ax.axis("off")
    plt.tight_layout(); plt.show()

draw_filters(W0_conv1, "conv1 filters -- before training (random)")
draw_filters(W1,       "conv1 filters -- after training")

print("權重改變幅度: 平均 |dW| = %.5f，最大 = %.5f" %
      (np.abs(W1 - W0_conv1).mean(), np.abs(W1 - W0_conv1).max()))

### 中間層看到的東西

把一張圖餵進去，看 `conv1` 的 16 個輸出（feature map）。
每一張都是「這個 filter 在圖上哪些位置有反應」。


In [ ]:
probe = tf.keras.Model(model.inputs, model.get_layer("conv1").output)
fmaps = probe.predict(x_test_f[:1], verbose=0)[0]     # (28,28,16)

fig, axes = plt.subplots(2, 9, figsize=(14, 3.4))
axes[0][0].imshow(x_test[0], cmap="gray")
axes[0][0].set_title("input = %d" % y_test[0], fontsize=9); axes[0][0].axis("off")
axes[1][0].axis("off")
for i in range(16):
    ax = axes[(i // 8), (i % 8) + 1]
    ax.imshow(fmaps[..., i], cmap="viridis")
    ax.set_title("f%d" % i, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 7. 它錯在哪裡？

平均準確率會騙人。真正有用的是**看它錯的那些**：是隨機錯，還是某兩個數字一直搞混？

混淆矩陣第 `i` 列第 `j` 行 = 「正解是 i、模型答 j」的張數。
對角線是答對的，其他格子是錯的。


In [ ]:
pred = model.predict(x_test_f, verbose=0)
pred_lbl = pred.argmax(axis=1)

cm = np.zeros((10, 10), dtype=int)
for t, p in zip(y_test, pred_lbl):
    cm[t][p] += 1

print("    " + "".join("%6d" % j for j in range(10)) + "   <- 模型答")
for i in range(10):
    print("%2d |" % i + "".join(("%6d" % cm[i][j]) if i != j else ("%5d*" % cm[i][j])
                                for j in range(10)))
print("（* 是對角線 = 答對）")
print()

off = [(cm[i][j], i, j) for i in range(10) for j in range(10) if i != j]
off.sort(reverse=True)
print("最常搞混的前 6 組:")
for n, i, j in off[:6]:
    print("  正解 %d 被答成 %d : %3d 次" % (i, j, n))

### 混淆矩陣熱圖

同樣的資料畫成圖。**對角線越亮越好**，對角線以外任何一個亮點都是一種系統性的錯誤。

下面右圖刻意把對角線挖掉（設成 0），這樣顏色範圍才不會被對角線那些幾千的數字壓扁 ——
不然其他格子全都是同一個顏色，什麼也看不出來。**這是畫混淆矩陣最常見的坑。**


In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.5))

im1 = a1.imshow(cm, cmap="Blues")
a1.set_title("confusion matrix (all)")
fig.colorbar(im1, ax=a1, fraction=0.046)

cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)          # 挖掉對角線，才看得到錯誤的細節
im2 = a2.imshow(cm_off, cmap="Reds")
a2.set_title("errors only (diagonal removed)")
fig.colorbar(im2, ax=a2, fraction=0.046)

for a in (a1, a2):
    a.set_xticks(range(10)); a.set_yticks(range(10))
    a.set_xlabel("predicted (模型答)"); a.set_ylabel("true (正解)")

# 把數字標上去
for i in range(10):
    for j in range(10):
        if cm[i][j]:
            a1.text(j, i, cm[i][j], ha="center", va="center", fontsize=7,
                    color="white" if cm[i][j] > cm.max()*0.5 else "black")
        if i != j and cm[i][j]:
            a2.text(j, i, cm[i][j], ha="center", va="center", fontsize=7,
                    color="white" if cm[i][j] > cm_off.max()*0.5 else "black")

plt.tight_layout(); plt.show()

# 每個類別各自的正確率 -- 平均值藏起來的東西在這裡
print("每個數字的正確率（recall）:")
for i in range(10):
    r = cm[i][i] / cm[i].sum()
    print("  %d : %.4f  %s" % (i, r, "#" * int(r * 50)))

### 預測 vs 預期

最直觀的一張：隨機抓 24 張測試圖，標題寫「正解 → 預測」。
**綠色 = 答對，紅色 = 答錯。**


In [ ]:
conf_all = pred.max(axis=1)          # 每張圖模型最高的那個機率 = 它的「信心」
rng = np.random.default_rng(0)
pick = rng.choice(len(x_test), 24, replace=False)

fig, axes = plt.subplots(3, 8, figsize=(14, 6))
for ax, i in zip(axes.ravel(), pick):
    ok = pred_lbl[i] == y_test[i]
    ax.imshow(x_test[i], cmap="gray")
    ax.set_title("%d -> %d  (%.0f%%)" % (y_test[i], pred_lbl[i], conf_all[i]*100),
                 fontsize=9, color="green" if ok else "red")
    ax.axis("off")
plt.suptitle("expected -> predicted   (green = correct, red = wrong)")
plt.tight_layout(); plt.show()

n_ok = int((pred_lbl[pick] == y_test[pick]).sum())
print("這 24 張裡答對 %d 張" % n_ok)

In [ ]:
# 把「很有信心但答錯」的圖找出來看 -- 這些通常人也會覺得難認
conf = pred.max(axis=1)
bad = np.where(pred_lbl != y_test)[0]
bad = bad[np.argsort(-conf[bad])][:10]

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for ax, i in zip(axes, bad):
    ax.imshow(x_test[i], cmap="gray")
    ax.set_title("%d->%d\n%.0f%%" % (y_test[i], pred_lbl[i], conf[i]*100), fontsize=8)
    ax.axis("off")
plt.suptitle("highest-confidence mistakes")
plt.tight_layout(); plt.show()

## 8. 先看一眼量化會怎樣

第 2 課才會真的量化，但先看數字你會比較有感。

量化 = 把 float32 權重壓成 uint8（每層一個 `scale`）。
一層的 `scale ≈ max|W| / 127`，也就是**這一層能表示的最小刻度**。

重點：`scale` 是被那一層的**極端值**決定的。如果某層有一兩個異常大的權重，
整層的刻度就被拉粗，其他權重全部一起受害 —— 這是量化掉準度的主要來源。


In [ ]:
print("%-8s %9s %12s %12s %16s" % ("層", "參數量", "min", "max", "scale=max|W|/127"))
for name in ["conv1", "conv2", "dense1", "dense2"]:
    w = model.get_layer(name).get_weights()[0]
    s = np.abs(w).max() / 127.0
    print("%-8s %9d %12.4f %12.4f %16.8f" % (name, w.size, w.min(), w.max(), s))

print()
w = model.get_layer("dense1").get_weights()[0].ravel()
hist, edges = np.histogram(w, bins=25)
for c, e in zip(hist, edges):
    print("  %+8.4f | %s" % (e, "#" * int(46.0 * c / hist.max())))
print()
print("dense1 權重分布 -- 絕大多數擠在 0 附近，但 scale 是被最外側那幾根決定的。")

## 9. 存檔，交給第 2 課

要產出兩樣東西：

### ① `mnist_cnn.h5` —— 模型本身

`include_optimizer=False` 很重要：optimizer 的狀態（Adam 的動量）只有繼續訓練才需要，
轉檔工具看到不認得的東西可能直接掛掉。

> **Keras 3 注意**：Colab 現在是 Keras 3，存出來的 `.h5` schema 跟 Keras 2 不同，
> acuity 有機會讀不開。如果第 2 課的 `import` 那步失敗，回來跑最後那格的 fallback。

### ② 校正圖 —— 量化用的

量化工具需要知道「真實輸入長什麼樣」才能決定每一層的數值範圍，所以要給它一批代表性的圖。

兩個關鍵：
- **每個類別取一樣多張**（這裡 10 類各 20 張）。只給數字 1 的話量化範圍會偏掉。
- 存成 PNG 配一個 `dataset.txt` 列出檔名 —— 這是 acuity 的約定格式。


In [ ]:
OUT = "mnist_out"
os.makedirs(OUT + "/calib", exist_ok=True)

H5 = OUT + "/mnist_cnn.h5"
model.save(H5, include_optimizer=False)
print("saved", H5, os.path.getsize(H5), "bytes")

# ---- 校正圖：每類 20 張，從 test set 取（沒被訓練看過）----
N_CALIB, PER = 200, 20
names, cnt, i = [], {}, 0
while len(names) < N_CALIB and i < len(x_test):
    lbl = int(y_test[i])
    if cnt.get(lbl, 0) < PER:
        cnt[lbl] = cnt.get(lbl, 0) + 1
        fn = "calib/d%d_%04d.png" % (lbl, i)
        png = tf.io.encode_png(to3ch(x_test[i:i+1])[0]).numpy()   # 三通道，跟訓練一致
        open(os.path.join(OUT, fn), "wb").write(png)
        names.append(fn)
    i += 1

open(OUT + "/dataset.txt", "w").write("\n".join(names) + "\n")
# acuity 的 mean/scale 檔：mean = 0,0,0   scale = 1/255
open(OUT + "/channel_mean_value.txt", "w").write("0 0 0 0.00392157\n")

print("校正圖 %d 張，每類 %s" % (len(names), sorted(cnt.items())))

ZIP = "mnist_calib.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, fs in os.walk(OUT):
        for f in fs:
            p = os.path.join(root, f)
            zf.write(p, os.path.relpath(p, OUT))
print("打包 ->", ZIP, os.path.getsize(ZIP), "bytes")

In [ ]:
from google.colab import files
files.download("mnist_calib.zip")   # 裡面含 mnist_cnn.h5 + calib/ + dataset.txt

### 只在第 2 課 import 失敗時才跑這格（Keras 2 相容存檔）

跑完**一定要 `Runtime -> Restart session`** —— `TF_USE_LEGACY_KERAS`
必須在 `import tensorflow` 之前設定才有效。重開後從第 0 格重跑一次。


In [ ]:
%pip -q install tf-keras
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
print("裝好了。現在去 Runtime -> Restart session，然後從第 0 格重跑。")

## 這一課的重點回顧

1. **先看資料再寫模型** —— 形狀、範圍、類別平衡，三件事親眼確認。
2. **訓練前先量起跑線** —— `loss ≈ ln(10)`、`acc ≈ 0.1`。沒有基準線就無法判斷有沒有在學。
3. **輸入契約是硬體逼出來的** —— 三通道是因為 `img_resize_planar()` 寫死 `W*H*3`；
   `/255` 的落差要靠 `inputmeta` 的 `scale` 橋接。
4. **`Reshape` 不是 `Flatten`** —— NPU 編譯期就要把記憶體配死，不吃動態 shape。
5. **平均準確率會騙人** —— 看混淆矩陣，看它錯在哪。
6. **量化的 `scale` 由極端值決定** —— 這是準確率掉下來的主因。

---

### 自己動手試（真的會學到東西的實驗）

- 把 `EPOCHS` 改成 `1` 從第 3 格重跑，比較準確率差多少。**大部分的學習發生在第一個 epoch。**
- 把 `Conv2D(16, ...)` 改成 `Conv2D(8, ...)`，看準確率掉多少、參數少多少。
- 第 4 格不要 compile 就直接 `fit`，看它報什麼錯。
- 把 `validation_split` 拿掉，你就再也看不出有沒有過擬合 —— 體會一下那種盲目感。

跑完把 `float test accuracy` 和混淆矩陣貼給我，我們進第 2 課：轉檔。
